In [2]:
import time

import torch
from torch import nn

from pract11.Guia11Reformat.src.ClassifierMLP import ClassifierMLP
from pract11.Guia11Reformat.src.Optimizer import create_optimizer
from pract11.Guia11Reformat.src.Trainer import Trainer
from pract11.Guia11Reformat.src.Visualizer import Visualizer
from src.Config import Config
from src.FashionMNISTDATA import FashionMNISTData


def runExperiment(config):
    # Configuración
    device = config.setup()
    # Datos
    data = FashionMNISTData(
        batch_size=config.BATCH_SIZE,
        seed=config.SEED, color=config.COLOR
    )

    if config.showSample:
        data.inspect_example()

    # Modelo
    torch.manual_seed(Config.SEED)

    model = ClassifierMLP(
        p_dropout=config.DROPOUT_P
    ).to(device)

    # Comprobar dimensiones
    if config.showDimentions:
        images, labels = next(iter(data.train_loader))

        with torch.no_grad():
            classifications = model(images.to(device))

        print(f"images.shape: {images.shape}")
        print(f"labels.shape: {labels.shape}")

        n_params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
        print(f"Parámetros entrenables: {n_params:,}")

        print("\nDimensiones:")
        print("Entrada:", images.shape)
        print("Logits:", classifications.shape)

    # Función de pérdida y optimizador
    loss_fn = nn.CrossEntropyLoss()

    optimizer = create_optimizer(
        model,
        config.OPTIMIZER_NAME
    )

    print("\nOptimizador:", config.OPTIMIZER_NAME)
    print("Dropout:", config.DROPOUT_P)

    # Untrained
    if config.showUntrained:
        Visualizer.showPredictions(model, device, data)

    # Entrenamiento
    trainer = Trainer(
        model,
        optimizer,
        loss_fn,
        device,
        config
    )
    start = time.perf_counter()
    trainer.fit(
        data.train_loader,
        data.valid_loader,
        epochs=config.EPOCHS
    )

    print(f"Training time: {time.perf_counter() - start:.2f} seconds")
    # Gráfico de pérdidas
    if config.showLoss: Visualizer.plot_history(trainer.history)

    # Evaluación final
    trainer.test(data.test_loader)

    #Trained
    if config.showTrained:
        Visualizer.showPredictions(model, device, data)
    return trainer


# ============================================================
# EJECUCIÓN
# ============================================================
# showSample=False, showDimentions=False, showUntrained=False, showLoss=False, showTrained=True, showDimentions2=False, showEpochData=True
runExperiment(Config(dropout_p=0.2, epochs=2,showInit=False))

# expies = {
#         "SGD": Config(dropout_p=0.2, epochs=2, optimizer_name="SGD", showSample=False, showDimentions=False,
#                         showDimentions2=False, showUntrained=False, showLoss=False, showTrained=False,
#                         showEpochData=False, showInit=False),
#           "Adam": Config(dropout_p=0.2, epochs=2, optimizer_name="Adam", showSample=False, showDimentions=False,
#                          showDimentions2=False, showUntrained=False, showLoss=False, showTrained=False,
#                          showEpochData=False, showInit=False)
#           }
#
# results = {}
# for title, exp in expies.items():
#     history = runExperiment(exp)
#     results[title] = history
# Visualizer.showExpieResults(results)



========== CONFIGURATION ==========
Seed:             42
Batch size:       100
Epochs:           2
Latent dimension: 64
Dropout:          0.2
Optimizer:        Adam

========== ENVIRONMENT ==========
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Cantidad de imágenes:
Train: 50000
Validation: 10000
Test: 10000

Ejemplo:
Shape: torch.Size([1, 28, 28])
Etiqueta: Ankle boot
Mínimo: 0.0
Máximo: 1.0
images.shape: torch.Size([100, 1, 28, 28])
labels.shape: torch.Size([100])
Parámetros entrenables: 109,386

Dimensiones:
Entrada: torch.Size([100, 1, 28, 28])
Logits: torch.Size([100, 10])

Optimizador: Adam
Dropout: 0.2

Iniciando entrenamiento... Opti -> Adam | Drop -> 0.2 | Batch size -> 100


KeyboardInterrupt: 